# START DAY-1.....

# Masked Language Modeling (MLM)
> The model hides some words and learns to fill them in.

Example

> Original sentence:

>>The cat sat on the mat.

>Training input:

>>The cat [MASK] on the mat.

>Target output:

>>sat

>Another example:

>>Input:  I drink [MASK] every morning.
Output: coffee

The model can use both sides of the missing word:

I drink [MASK] every morning. {[Mask left & rigt works seeing after predict]}
     

So it understands the sentence by looking at the complete surrounding context.

Think of it like a fill-in-the-blank exam:
The capital of France is ______.

You know the answer is Paris because you can read the whole sentence.

# Causal Language Modeling (CLM)
The model predicts the next word based only on what has already appeared.

Example

>Training sequence:

>>The cat sat on the mat.

The model learns:

>>Input:  The
>>Output: cat

>>Input:  The cat
>>Output: sat

>>Input:  The cat sat
>>Output: on

>>Input:  The cat sat on
>>Output: the

>>Input:  The cat sat on the
>>Output: mat

When generating text:

>Prompt:
The weather today is

>Model predicts:
sunny

>Then:
The weather today is sunny

>Next prediction:
and

>Then:
The weather today is sunny and

>Next prediction:
warm

The model cannot see future words while predicting.

# Pretrain Model In Deep Learning ResNet(CV) model....

In [38]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np

In [39]:
model = ResNet50(weights='imagenet')

In [40]:
img_path = '/content/drive/MyDrive/SunnySavita-LLMCourseCodes/ LLM Fine-Tuning: 02 Understanding Model Pretraining and Training in AI/assests/DogImage.jpg'

In [41]:
img = image.load_img(img_path, target_size=(224, 224))

In [42]:
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

In [43]:
preds = model.predict(x)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [44]:
preds

array([[5.47578718e-11, 8.45022524e-11, 8.16548495e-11, 3.94661470e-10,
        8.99564659e-11, 1.80616841e-10, 2.54706551e-11, 1.70061814e-08,
        4.90613861e-09, 1.01817499e-09, 2.12406218e-10, 2.56907984e-11,
        1.23364166e-10, 1.84276525e-10, 1.33123321e-11, 1.45848777e-09,
        5.85575977e-10, 1.26939370e-09, 8.68999539e-10, 9.97822380e-11,
        5.42555272e-11, 1.62384872e-09, 2.32725176e-11, 1.94952565e-09,
        1.55951485e-09, 1.82369220e-09, 4.54774468e-10, 4.30443708e-10,
        1.33218131e-10, 1.31229361e-10, 9.88444049e-11, 1.88177529e-10,
        3.86839838e-10, 1.56714230e-09, 3.43115508e-10, 1.51119628e-09,
        4.22018913e-11, 4.17230583e-10, 2.31340613e-09, 1.64054759e-09,
        1.10462146e-10, 2.60056005e-10, 5.59245850e-09, 3.16481485e-09,
        1.37021852e-10, 2.92404714e-08, 3.89588417e-10, 6.40705444e-09,
        2.14256363e-10, 8.13033529e-10, 2.10849768e-10, 3.94382305e-10,
        5.37936140e-10, 8.21633733e-11, 1.80747539e-09, 2.248159

In [45]:
for _, label, prob in decode_predictions(preds, top=10)[0] :
  print(f'{label} : {prob:.4f}')

German_shepherd : 0.9964
malinois : 0.0035
Airedale : 0.0000
muzzle : 0.0000
briard : 0.0000
Rottweiler : 0.0000
Australian_terrier : 0.0000
kelpie : 0.0000
soccer_ball : 0.0000
bloodhound : 0.0000


# Below code for the BERT Model.....

In [46]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F

In [47]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
text = "The capital city of France is [MASK]."

In [49]:
inputs = tokenizer(text, return_tensors="pt")

In [50]:
# Forward pass (no gradients needed for interface)
with torch.inference_mode() :
  logits = model(**inputs).logits   # shape : [batch, seq_len, vocab]

In [51]:
# Find the position of the [MASK] token
mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

In [52]:
# Soft-max over vocab + take top-k predictions for that position
probs = F.softmax(logits[0, mask_idx], dim = -1)   # [1, vocab]
top_k = torch.topk(probs, k=5, dim = -1)           # top-5 guesses

In [53]:
# Decode & print
print("Top-5 predictions for [MASK] :")
for token_id, prob in zip(top_k.indices[0], top_k.values[0]) :
  token = tokenizer.decode([token_id])
  print(f"{token:<12} -> {prob.item():.4f}")

Top-5 predictions for [MASK] :
paris        -> 0.3465
lille        -> 0.0925
lyon         -> 0.0850
marseille    -> 0.0623
toulouse     -> 0.0405


# END DAY-1

In [56]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

#Model name : BERT fine tuned for classification
model_name = "textattack/bert-base-uncased-SST-2"

#load manually
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

#Create pipline
classifier = pipeline("sentiment-analysis", model = model, tokenizer = tokenizer)

#predict
result = classifier("The movie was a masterpiece. I love it!")
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.9996596574783325}]
